# 🏈 Fantasy Football Draft Analysis (2024–2025)
*Justin McKendry · August 2025*

---

## 📚 Project Overview

This notebook analyzes player performance, draft value, and team efficiency based on our league's draft and season data.  
We explore who maximized their draft capital, which picks over- or under-performed, and how VORP (Value Over Replacement Player) relates to league standings.

**Key Metrics:**
- 📈 VORP (custom-calculated per position)
- 🎯 Draft Delta (Actual Pick - ADP)
- 💥 Boom/Bust Score (Actual - Projected points)

--- 
## 🧪 2. Data Processing

### 🔍 Purpose
Take the data collected in NB01:
- Clean each dataframe
- Filter to input into database
- Create database
- Populate database

# Importing Packages

In [1]:
import pandas as pd 
import os
import ast
from pathlib import Path
import sqlite3
import re
from rapidfuzz import process, fuzz
from utils import clean_name, read_csvs, generate_schema_from_df



# Visualise the distribution of comments per post
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from datetime import datetime
from sqlalchemy import create_engine, text
from config import CURRENT_SEASON, LINEUP_SLOT_MAP, NEXT_SEASON

# Import our custom Reddit API module

# --- Configuration for Jupyter ---
# The following magic command is for Jupyter notebooks to render plots inline.
# It should be commented out when running as a standalone script.
%config InlineBackend.figure_formats = ['svg']



# Get the data from the csvs

In [2]:
foreign_keys = {
    "player_id": "players(player_id)",
}

next_season_csv = f'../data/raw/{NEXT_SEASON}/projections/espn_proj/{NEXT_SEASON}_proj_stats.csv'
if os.path.exists(next_season_csv):
    next_season_database_df = pd.read_csv(next_season_csv)
else:
    fallback_year = NEXT_SEASON - 1
    print(
        f"No projections CSV yet for {NEXT_SEASON} at {next_season_csv} - run NB01 once "
        f"ESPN publishes {NEXT_SEASON} projections/ADP (usually a few weeks before the draft). "
        f"Falling back to {fallback_year}'s projections, relabeled as {NEXT_SEASON}, so the rest "
        f"of the pipeline (and the live draft tool) still runs end-to-end for local dev/demo."
    )
    next_season_database_df = pd.read_csv(
        f'../data/raw/{fallback_year}/projections/espn_proj/{fallback_year}_proj_stats.csv'
    )
    next_season_database_df['year'] = NEXT_SEASON

next_season_database_df.head()

No projections CSV yet for 2026 at ../data/raw/2026/projections/espn_proj/2026_proj_stats.csv - run NB01 once ESPN publishes 2026 projections/ADP (usually a few weeks before the draft). Falling back to 2025's projections, relabeled as 2026, so the rest of the pipeline (and the live draft tool) still runs end-to-end for local dev/demo.


,Unnamed: 0,player_name,player_id,pro_team,projected_points,year,eligible_slots,proj_defensive0PointsAllowed,proj_defensive1To6PointsAllowed,proj_defensive7To13PointsAllowed,...,proj_passing50PlusYardTD,proj_passing300To399YardGame,proj_passing400PlusYardGame,proj_passing2PtConversions,proj_passingInterceptions,proj_passingCompletionPercentage,proj_passingTimesSacked,proj_65,proj_69,proj_211
0,0,George Kittle,3040151,SF,226.90,2026,"['WR/TE', 'TE', 'RB/WR/TE', 'OP', 'BE', 'IR']",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,DK Metcalf,4047650,PIT,240.35,2026,"['RB/WR', 'WR', 'WR/TE', 'RB/WR/TE', 'OP', 'BE...",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Alvin Kamara,3054850,NO,265.78,2026,"['RB', 'RB/WR', 'RB/WR/TE', 'OP', 'BE', 'IR']",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Quinshon Judkins,4685702,CLE,48.90,2026,"['Rookie', 'RB', 'RB/WR', 'RB/WR/TE', 'OP', 'B...",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,Bills D/ST,-16002,BUF,111.93,2026,"['D/ST', 'BE', 'IR']",0.011102,0.05001,0.179036,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Setting the root

df_adp = read_csvs("adp", years=range(2022, NEXT_SEASON + 1))

df_adp

⚠️ File not found for year 2026: ../data/raw/2026/adp/FantasyPros_2026_Overall_ADP_Rankings.csv


,rank,player_name,team_name,bye,POS,ESPN,Sleeper,avg,position,pos_rank,year,NFL,RTSports,FFC
0,1.0,Jonathan Taylor,IND,14,RB1,1.0,1.0,1.0,RB,1.0,2022,NaN,NaN,NaN
1,2.0,Christian McCaffrey,SF,9,RB2,2.0,2.0,2.0,RB,2.0,2022,NaN,NaN,NaN
2,3.0,Derrick Henry,BAL,10,RB3,5.0,3.0,4.0,RB,3.0,2022,NaN,NaN,NaN
3,4.0,Cooper Kupp,SEA,11,WR1,3.0,5.0,4.0,WR,1.0,2022,NaN,NaN,NaN
4,5.0,Austin Ekeler,WAS,14,RB4,4.0,4.0,4.0,RB,4.0,2022,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2330,893.0,Jacardia Wright,SEA,8.0,RB207,NaN,NaN,848.0,RB,207.0,2025,848.0,NaN,NaN
2331,894.0,Jacoby Jones,WAS,12.0,LB2,NaN,NaN,849.0,LB,2.0,2025,849.0,NaN,NaN
2332,895.0,Mark McNamee,GB,5.0,K62,NaN,NaN,850.0,K,62.0,2025,850.0,NaN,NaN
2333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025,NaN,NaN,NaN


In [4]:
df_players = read_csvs("league_stats", "player_stats", range(2020, CURRENT_SEASON + 1))

df_players
cols_to_front = ['player_name', 'player_id','year','team_name','eligible_slots']

# Reorder
df_players = df_players[cols_to_front + [c for c in df_players.columns if c not in cols_to_front]]
df_players = df_players.fillna(0)
df_players[df_players['year'] == 2024]

,player_name,player_id,year,team_name,eligible_slots,points,avg_points,projected_points,projected_avg_points,actual_rushingAttempts,...,proj_214,proj_215,proj_216,proj_221,proj_227,proj_233,proj_212,proj_213,proj_211,proj_112
1910,Saquon Barkley,3929630,2024,Stretchy Pups Elite,"['RB', 'RB/WR', 'RB/WR/TE', 'OP', 'BE', 'IR']",341.0,21.31,251.98,18.00,345.0,...,0.0,0.0,0.0,0.0,0.0,0.0,59.245452,20.259327,0.000000,0.0
1911,De'Von Achane,4429160,2024,Stretchy Pups Elite,"['RB', 'RB/WR', 'RB/WR/TE', 'OP', 'BE', 'IR']",286.0,16.82,210.02,15.00,203.0,...,0.0,0.0,0.0,0.0,0.0,0.0,41.651158,21.286767,0.000000,0.0
1912,Anthony Richardson,4429084,2024,Stretchy Pups Elite,"['QB', 'OP', 'BE', 'IR']",153.0,13.91,274.87,18.32,86.0,...,0.0,0.0,0.0,0.0,0.0,0.0,34.037845,0.000000,164.057207,0.0
1913,Jaylen Waddle,4372016,2024,Stretchy Pups Elite,"['RB/WR', 'WR', 'WR/TE', 'RB/WR/TE', 'OP', 'BE...",142.0,9.47,217.93,14.53,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.341483,48.628614,0.000000,0.0
1914,Malik Nabers,4595348,2024,Stretchy Pups Elite,"['Rookie', 'RB/WR', 'WR', 'WR/TE', 'RB/WR/TE',...",266.0,17.73,221.80,14.79,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,49.570501,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2429,Brenden Rice,4686781,2024,FA,"['Rookie', 'RB/WR', 'WR', 'WR/TE', 'RB/WR/TE',...",0.0,0.00,12.58,0.84,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.831978,0.000000,0.0
2430,Trenton Irwin,3931391,2024,FA,"['RB/WR', 'WR', 'WR/TE', 'RB/WR/TE', 'OP', 'BE...",3.0,0.43,57.03,3.80,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,11.537681,0.000000,0.0
2431,Velus Jones Jr.,4035693,2024,FA,"['RB', 'RB/WR', 'RB/WR/TE', 'OP', 'BE', 'IR']",0.0,0.00,6.24,0.42,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.726371,-0.174762,0.000000,0.0
2432,Quintin Morris,4244049,2024,FA,"['WR/TE', 'TE', 'RB/WR/TE', 'OP', 'BE', 'IR']",14.0,0.88,8.66,0.62,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.899611,0.000000,0.0


In [5]:
df_teams = read_csvs(data="teams_data", years=range(2020, CURRENT_SEASON + 1))


In [6]:
df_draft = read_csvs(data="draft_data", years=range(2020, CURRENT_SEASON + 1))
df_draft

,autoDraftTypeId,id,lineupSlotId,memberId,overallPickNumber,player_id,roundId,roundPickNumber,team_id,year,league_id
0,0,1,0,{REDACTED-ESPN-MEMBER-ID},1,3916387,1,1,11,2020,780575
1,0,2,2,{REDACTED-ESPN-MEMBER-ID},2,3929630,1,2,1,2020,780575
2,0,3,2,{REDACTED-ESPN-MEMBER-ID},3,3051392,1,3,9,2020,780575
3,0,4,2,{REDACTED-ESPN-MEMBER-ID},4,3117251,1,4,3,2020,780575
4,0,5,2,{REDACTED-ESPN-MEMBER-ID},5,4242214,1,5,14,2020,780575
...,...,...,...,...,...,...,...,...,...,...,...
1163,0,220,20,{REDACTED-ESPN-MEMBER-ID},220,4045163,16,10,16,2025,780575
1164,0,221,17,{REDACTED-ESPN-MEMBER-ID},221,16339,16,11,8,2025,780575
1165,3,222,20,NaN,222,4426386,16,12,15,2025,780575
1166,0,223,17,{REDACTED-ESPN-MEMBER-ID},223,4566192,16,13,6,2025,780575


# Clean Data to Input to SQL Database

### Cleaning df_players

Calculating games_played which is the sum of actual_teamLoss and actual_teamWin. Both those variables are calculations of how their
pro team did WHEN THE PLAYER PLAYED

In [7]:

df_players['games_played'] = df_players['actual_teamLoss'] + df_players['actual_teamWin']
df_players

,player_name,player_id,year,team_name,eligible_slots,points,avg_points,projected_points,projected_avg_points,actual_rushingAttempts,...,proj_215,proj_216,proj_221,proj_227,proj_233,proj_212,proj_213,proj_211,proj_112,games_played
0,Jonathan Taylor,4242335,2020,Stretchy Pups,"['Rookie', 'RB', 'RB/WR', 'RB/WR/TE', 'OP', 'B...",205.0,13.67,154.47,11.03,232.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,15.0
1,Henry Ruggs III,4241475,2020,Stretchy Pups,"['Rookie', 'RB/WR', 'WR', 'WR/TE', 'RB/WR/TE',...",51.0,3.92,101.57,6.77,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,13.0
2,Brandin Cooks,16731,2020,Stretchy Pups,"['RB/WR', 'WR', 'WR/TE', 'RB/WR/TE', 'OP', 'BE...",145.0,9.67,104.57,8.04,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,15.0
3,Ben Roethlisberger,5536,2020,Stretchy Pups,"['QB', 'OP', 'BE', 'IR']",259.0,17.27,240.27,17.16,25.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,15.0
4,Justin Jefferson,4262921,2020,Stretchy Pups,"['Rookie', 'RB/WR', 'WR', 'WR/TE', 'RB/WR/TE',...",179.0,11.19,87.16,5.81,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2950,Jackson Hawes,4573699,2025,FA,"['Rookie', 'WR/TE', 'TE', 'RB/WR/TE', 'OP', 'B...",46.0,2.71,3.65,0.21,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.206831,0.0,0.0,17.0
2951,Will Levis,4361418,2025,FA,"['QB', 'OP', 'BE', 'IR']",0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2952,Tommy DeVito,4240391,2025,FA,"['QB', 'OP', 'BE', 'IR']",0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2953,Leonard Fournette,3115364,2025,FA,"['RB', 'RB/WR', 'RB/WR/TE', 'OP', 'BE', 'IR']",0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


### Cleaning df_adp

df_adp contains the average draft position of players across all fantasy football leagues. Our league specifically uses the ESPN app but Sleeper is also used to calculate the average. This can be used as a marker to see if our league draft data is skewed in a certain way.

#### Getting the position rank


Currently this code block extracts the position and the posRank from the POS column. The POS column contains a number like WR2. This means that that player is on average, the second WR to get drafted. It's simpiler for analysis later on if the position and rank are seperate features.

#### Map the Defenses to match the leagues naming convention
In our league, defenses are identified with the team name and D/ST 

i.e.
**Chicago Bears = Bears D/ST** 

I need to map the names to get a consistent formating

#### Match the player names to their league player_id

As adp is pulled from an outside source, their players do not have the same player_id numbers. I have to match the df_adp names to the df_players names to assign them the propper league player_id numbers. Unfortunately both dataframes use different formatting. Suffixes are included in df_adp but not in df_players. I had to remove them from df_adp before matching as well as remove all capitalization just in case. I created a player_clean column in d_adp and matched that to df_players player_clean column. Most players were matched, however, df_adp had more players than df_players so some in df_adp were not assigned a player_id

In [8]:
# Step 4: Clean names
df_adp["player_clean"] = df_adp["player_name"].apply(clean_name)
df_players["player_clean"] = df_players["player_name"].apply(clean_name)

# Step 5: Create mapping and apply
df_adp = df_adp.merge(
    df_players[["player_id", "player_clean"]],
    how="left",
    on="player_clean"
)

df_players = df_players.drop(columns='player_clean')
df_adp = df_adp.drop(columns='player_clean')



In [9]:
df_adp['player_id'] = pd.to_numeric(df_adp['player_id'], errors='coerce').astype('Int64')

df_adp = df_adp.drop_duplicates(subset=["year","player_id"], keep="first")
df_adp.fillna(0)

,rank,player_name,team_name,bye,POS,ESPN,Sleeper,avg,position,pos_rank,year,NFL,RTSports,FFC,player_id
0,1.0,Jonathan Taylor,IND,14,RB1,1.0,1.0,1.0,RB,1.0,2022,0.0,0.0,0.0,4242335
6,2.0,Christian McCaffrey,SF,9,RB2,2.0,2.0,2.0,RB,2.0,2022,0.0,0.0,0.0,3117251
12,3.0,Derrick Henry,BAL,10,RB3,5.0,3.0,4.0,RB,3.0,2022,0.0,0.0,0.0,3043078
18,4.0,Cooper Kupp,SEA,11,WR1,3.0,5.0,4.0,WR,1.0,2022,0.0,0.0,0.0,2977187
24,5.0,Austin Ekeler,WAS,14,RB4,4.0,4.0,4.0,RB,4.0,2022,0.0,0.0,0.0,3068267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7869,867.0,Kalel Mullings,TEN,10.0,RB200,0.0,0.0,819.0,RB,200.0,2025,819.0,0.0,0.0,4429121
7875,873.0,Konata Mumpfield,LAR,8.0,WR299,0.0,0.0,826.0,WR,299.0,2025,826.0,0.0,0.0,4710855
7880,878.0,Max Brosmer,MIN,6.0,QB123,0.0,0.0,831.0,QB,123.0,2025,831.0,0.0,0.0,4573398
7881,879.0,Myles Price,MIN,6.0,WR302,0.0,0.0,832.0,WR,302.0,2025,832.0,0.0,0.0,4430656


In [10]:
unmatched_players = df_adp[df_adp["player_id"].isna()]
unmatched_players.to_csv("../data/raw/other/unmatched.csv", index=False)

In [11]:
df_players[df_players['year'] == 2024] 


,player_name,player_id,year,team_name,eligible_slots,points,avg_points,projected_points,projected_avg_points,actual_rushingAttempts,...,proj_215,proj_216,proj_221,proj_227,proj_233,proj_212,proj_213,proj_211,proj_112,games_played
1910,Saquon Barkley,3929630,2024,Stretchy Pups Elite,"['RB', 'RB/WR', 'RB/WR/TE', 'OP', 'BE', 'IR']",341.0,21.31,251.98,18.00,345.0,...,0.0,0.0,0.0,0.0,0.0,59.245452,20.259327,0.000000,0.0,16.0
1911,De'Von Achane,4429160,2024,Stretchy Pups Elite,"['RB', 'RB/WR', 'RB/WR/TE', 'OP', 'BE', 'IR']",286.0,16.82,210.02,15.00,203.0,...,0.0,0.0,0.0,0.0,0.0,41.651158,21.286767,0.000000,0.0,17.0
1912,Anthony Richardson,4429084,2024,Stretchy Pups Elite,"['QB', 'OP', 'BE', 'IR']",153.0,13.91,274.87,18.32,86.0,...,0.0,0.0,0.0,0.0,0.0,34.037845,0.000000,164.057207,0.0,11.0
1913,Jaylen Waddle,4372016,2024,Stretchy Pups Elite,"['RB/WR', 'WR', 'WR/TE', 'RB/WR/TE', 'OP', 'BE...",142.0,9.47,217.93,14.53,4.0,...,0.0,0.0,0.0,0.0,0.0,3.341483,48.628614,0.000000,0.0,15.0
1914,Malik Nabers,4595348,2024,Stretchy Pups Elite,"['Rookie', 'RB/WR', 'WR', 'WR/TE', 'RB/WR/TE',...",266.0,17.73,221.80,14.79,5.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,49.570501,0.000000,0.0,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2429,Brenden Rice,4686781,2024,FA,"['Rookie', 'RB/WR', 'WR', 'WR/TE', 'RB/WR/TE',...",0.0,0.00,12.58,0.84,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,1.831978,0.000000,0.0,3.0
2430,Trenton Irwin,3931391,2024,FA,"['RB/WR', 'WR', 'WR/TE', 'RB/WR/TE', 'OP', 'BE...",3.0,0.43,57.03,3.80,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,11.537681,0.000000,0.0,7.0
2431,Velus Jones Jr.,4035693,2024,FA,"['RB', 'RB/WR', 'RB/WR/TE', 'OP', 'BE', 'IR']",0.0,0.00,6.24,0.42,3.0,...,0.0,0.0,0.0,0.0,0.0,2.726371,-0.174762,0.000000,0.0,3.0
2432,Quintin Morris,4244049,2024,FA,"['WR/TE', 'TE', 'RB/WR/TE', 'OP', 'BE', 'IR']",14.0,0.88,8.66,0.62,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.899611,0.000000,0.0,16.0


#### Converting player_id to Integer

The player_id column in df_adp was a float but in order to merge between tables it needed to be an int.

#### Renaming columns 

Need to rename columns to match the conventions of the database


## Cleaning df_draft

### Map positions in df_draft

This allows us to fill in missing position data. Position data was not included in the df_draft just the lineupSlotId. I had to reverse engineer the positions based off their lineupSlotIds. 

In [12]:
## RB = 2, WR = 4, WR = 23, QB = 20, TE = 6, K =17, D/ST = 16
df_draft['position'] = df_draft['lineupSlotId'].map(LINEUP_SLOT_MAP)
df_draft.columns

Index(['autoDraftTypeId', 'id', 'lineupSlotId', 'memberId',
       'overallPickNumber', 'player_id', 'roundId', 'roundPickNumber',
       'team_id', 'year', 'league_id', 'position'],
      dtype='object')

# Filtering Columns in Each Dataframe

Many columns will likely not be used, especially in the df_player. That contains around 100 different features, most of them being null values. 

In [13]:
current_team_info = (
    df_players[df_players["year"] == CURRENT_SEASON]
    .loc[:, ["player_id", "pro_team", "team_id", "team_name"]]
    .drop_duplicates(subset="player_id")
)
current_team_info = current_team_info.rename(columns={"team_id":"current_team_id", 
                                  "team_name": "current_team_name"})

players_df = (
    df_players
    .loc[:, ["player_id", "player_name"]]
    .drop_duplicates(subset="player_id")
)
players_database_df = players_df.merge(current_team_info, on="player_id", how="left")
stats_database_df = df_players

In [14]:
#players_database_df = pd.concat(
#    [players_database_df.assign(rookie=0),  # mark vets
#     rookies],
#    ignore_index=True
#)
#players_database_df[players_database_df['rookie'] == 1]

In [15]:
players_df

,player_id,player_name
0,4242335,Jonathan Taylor
1,4241475,Henry Ruggs III
2,16731,Brandin Cooks
3,5536,Ben Roethlisberger
4,4262921,Justin Jefferson
...,...,...
2935,4430870,KeAndre Lambert-Smith
2943,4040612,Luke Farrell
2945,4685397,Jordan James
2949,4565185,Tai Felton


In [16]:
## Extract just the columns needed from the dataframe to prepare to 
## setup the database


adp_columns = [
    'player_name', 
    'team_name', 
    'ESPN', 
    'position',
    'year',
    'Sleeper',
    'POS',
    'pos_rank',
    'avg',  
    'player_id'
]

draft_columns = [
    'autoDraftTypeId', 
    'id', 
    'lineupSlotId',
    'overallPickNumber', 
    'position',
    'player_id', 
    'roundId', 
    'roundPickNumber',
    'team_id',
    'year'
]

### Creating the Final dataframes

These are the final dataframes that will be used in the SQL database

In [17]:
adp_database_df = df_adp[adp_columns]
draft_database_df = df_draft[draft_columns]
teams_database_df = df_teams

# DataBase Creation

Creating the four different tables.

**draft** = league specific draft data

**players** = professional players and their stats

**teams** = teams within the league

**adp** = worldwide average draft position

### Creating the engine

In [18]:
engine = create_engine("sqlite:///../data/fantasy_data.db")

### Defining the Schema

In [19]:
draft_database_df.columns

Index(['autoDraftTypeId', 'id', 'lineupSlotId', 'overallPickNumber',
       'position', 'player_id', 'roundId', 'roundPickNumber', 'team_id',
       'year'],
      dtype='object')

In [20]:
draft_database_df

,autoDraftTypeId,id,lineupSlotId,overallPickNumber,position,player_id,roundId,roundPickNumber,team_id,year
0,0,1,0,1,QB,3916387,1,1,11,2020
1,0,2,2,2,RB,3929630,1,2,1,2020
2,0,3,2,3,RB,3051392,1,3,9,2020
3,0,4,2,4,RB,3117251,1,4,3,2020
4,0,5,2,5,RB,4242214,1,5,14,2020
...,...,...,...,...,...,...,...,...,...,...
1163,0,220,20,220,BE,4045163,16,10,16,2025
1164,0,221,17,221,K,16339,16,11,8,2025
1165,3,222,20,222,BE,4426386,16,12,15,2025
1166,0,223,17,223,K,4566192,16,13,6,2025


In [21]:
stats_drop_colums = [
    # All actual_### and proj_### that are just IDs (keep none of them unless mapped)
    'actual_2PtConversions', 'proj_2PtConversions',  # Already covered in touchdowns & points
    'actual_5','actual_6','actual_7','actual_8','actual_9','actual_10','actual_11','actual_12','actual_13','actual_14',
    'proj_5','proj_6','proj_7','proj_8','proj_9','proj_10','proj_11','proj_12',
    'actual_27','actual_28','actual_29','actual_30','actual_31','actual_32','actual_33','actual_34',
    'proj_27','proj_28','proj_29','proj_30','proj_31','proj_33','proj_34',
    'actual_47','actual_48','actual_49','actual_50','actual_51','actual_52','actual_54','actual_55',
    'proj_47','proj_48','proj_49','proj_50','proj_51','proj_54','proj_55',
    'actual_65','proj_65','proj_69',
    'actual_66','actual_67','actual_69','proj_66','proj_67',
    'actual_70','proj_70','proj_71','actual_71',
    'actual_100','proj_100',
    'actual_110','actual_111','actual_112','proj_110','proj_111','proj_112',
    'actual_116','actual_117','proj_116','proj_117',
    'actual_119','proj_119','proj_126','proj_137',
    'actual_143','actual_144','proj_143','proj_144',
    'actual_175','actual_176','actual_177','actual_178',
    'actual_179','actual_180','actual_181','actual_182','proj_198','proj_199','proj_200',
    'actual_183','actual_184','actual_185','proj_210','proj_221','proj_227','proj_233',
    'actual_186','actual_188','actual_189','actual_190','actual_191','actual_192','actual_193','actual_194','actual_195',
    'actual_196','actual_198','actual_199','actual_200',
    'actual_211','actual_212','actual_213','actual_214','actual_215','actual_216','actual_217','actual_218',
    'actual_219','actual_220','actual_221','actual_222','actual_223','actual_224','actual_225','actual_226',
    'actual_227','actual_228','actual_229','actual_230','actual_231','actual_232','actual_233','actual_234',
    'proj_211','proj_212','proj_213','proj_214','proj_215','proj_216','schedule',        # ESPN schedule object — not needed
        # Only needed if you're enforcing lineup logic, not for projections
    'acquisition_type' # Waiver/free agent status, not predictive
]


In [22]:
stats_database_df = stats_database_df.drop(columns=stats_drop_colums)


In [23]:
stats_database_df['eligible_slots']

0       ['Rookie', 'RB', 'RB/WR', 'RB/WR/TE', 'OP', 'B...
1       ['Rookie', 'RB/WR', 'WR', 'WR/TE', 'RB/WR/TE',...
2       ['RB/WR', 'WR', 'WR/TE', 'RB/WR/TE', 'OP', 'BE...
3                                ['QB', 'OP', 'BE', 'IR']
4       ['Rookie', 'RB/WR', 'WR', 'WR/TE', 'RB/WR/TE',...
                              ...                        
2950    ['Rookie', 'WR/TE', 'TE', 'RB/WR/TE', 'OP', 'B...
2951                             ['QB', 'OP', 'BE', 'IR']
2952                             ['QB', 'OP', 'BE', 'IR']
2953        ['RB', 'RB/WR', 'RB/WR/TE', 'OP', 'BE', 'IR']
2954        ['WR/TE', 'TE', 'RB/WR/TE', 'OP', 'BE', 'IR']
Name: eligible_slots, Length: 2955, dtype: object

In [24]:
import json

bad_slots = ['BE', 'IR', 'OP']

def drop_slots(slot_str):
    slot_list = ast.literal_eval(slot_str)
    # Return only the slots that are not in bad_slots
    return [slot for slot in slot_list if slot not in bad_slots]

stats_database_df['eligible_slots'] = stats_database_df['eligible_slots'].apply(drop_slots)
stats_database_df['eligible_slots'] = stats_database_df['eligible_slots']


In [25]:

stats_database_df["eligible_slots"] = stats_database_df["eligible_slots"].apply(
    lambda v: json.dumps(v if isinstance(v, list) else [])
)
stats_database_df['eligible_slots'].dtype

dtype('O')

In [26]:
foreign_keys = {
    "player_id": "players(player_id)",
    "team_id": "teams(team_id)",
}
players_stats_schema = generate_schema_from_df(
    stats_database_df,
    "players_stats",
    foreign_keys=foreign_keys)

print(players_stats_schema)

CREATE TABLE players_stats (
    "player_name" TEXT,
    "player_id" INTEGER,
    "year" INTEGER,
    "team_name" TEXT,
    "eligible_slots" TEXT,
    "points" DECIMAL(10, 4),
    "avg_points" DECIMAL(10, 4),
    "projected_points" DECIMAL(10, 4),
    "projected_avg_points" DECIMAL(10, 4),
    "actual_rushingAttempts" DECIMAL(10, 4),
    "actual_rushingYards" DECIMAL(10, 4),
    "actual_rushingTouchdowns" DECIMAL(10, 4),
    "actual_rushing40PlusYardTD" DECIMAL(10, 4),
    "actual_rushing50PlusYardTD" DECIMAL(10, 4),
    "actual_rushing100To199YardGame" DECIMAL(10, 4),
    "actual_rushing200PlusYardGame" DECIMAL(10, 4),
    "actual_rushingYardsPerAttempt" DECIMAL(10, 4),
    "actual_receivingReceptions" DECIMAL(10, 4),
    "actual_receivingYards" DECIMAL(10, 4),
    "actual_receivingTouchdowns" DECIMAL(10, 4),
    "actual_receivingTargets" DECIMAL(10, 4),
    "actual_receivingYardsAfterCatch" DECIMAL(10, 4),
    "actual_receivingYardsPerReception" DECIMAL(10, 4),
    "actual_fumbles" D

In [27]:
import sys
sys.path.append(os.path.abspath('..'))
from src.scoring import normalize_position
from config import POSITIONS

# next_season_database_df has no `position` column (ESPN's projections export
# doesn't include one) - derive it from the most recently known position for
# each player across our cleaned ADP and player-stats history. Both source
# tables also contain non-position labels mixed into that column (bench
# slot "BE", multi-slot eligibility strings like "RB/WR/TE", and a stray "0"
# in some older rows) which must be filtered out before picking "the most
# recent" value, otherwise a player's *last* row might be a garbage label
# instead of their real position.
valid_positions = set(POSITIONS) | {"D/ST"}
position_lookup = pd.concat([
    df_adp[['player_id', 'year', 'position']],
    df_players[['player_id', 'year', 'position']],
], ignore_index=True).dropna(subset=['player_id', 'position'])
position_lookup['player_id'] = pd.to_numeric(position_lookup['player_id'], errors='coerce')
position_lookup['position'] = position_lookup['position'].map(normalize_position)
position_lookup = position_lookup[position_lookup['position'].isin(valid_positions)]
position_lookup = (
    position_lookup.sort_values('year', ascending=False)
    .drop_duplicates(subset='player_id', keep='first')[['player_id', 'position']]
)

next_season_database_df['player_id'] = pd.to_numeric(next_season_database_df['player_id'], errors='coerce')
next_season_database_df = next_season_database_df.merge(position_lookup, on='player_id', how='left')
print(f"{next_season_database_df['position'].isna().sum()} players with no mappable position "
      f"(e.g. incoming rookies with no prior-season/ADP history)")

91 players with no mappable position (e.g. incoming rookies with no prior-season/ADP history)


In [28]:
next_season_cols = ['player_id', 'player_name', 'position', 'pro_team', 'projected_points', 'year']
next_season_database_df = next_season_database_df[next_season_cols].dropna(subset=['player_id', 'position'])
next_season_database_df['player_id'] = next_season_database_df['player_id'].astype(int)

next_season_schema = generate_schema_from_df(
    next_season_database_df,
    "next_season_projections",
    foreign_keys=foreign_keys)
print(next_season_schema)
next_season_database_df.head()

CREATE TABLE next_season_projections (
    "player_id" INTEGER,
    "player_name" TEXT,
    "position" TEXT,
    "pro_team" TEXT,
    "projected_points" DECIMAL(10, 4),
    "year" INTEGER,
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
);


,player_id,player_name,position,pro_team,projected_points,year
0,3040151,George Kittle,TE,SF,226.90,2026
1,4047650,DK Metcalf,WR,PIT,240.35,2026
2,3054850,Alvin Kamara,RB,NO,265.78,2026
3,4685702,Quinshon Judkins,RB,CLE,48.90,2026
4,-16002,Bills D/ST,DST,BUF,111.93,2026


In [29]:
# Define our database schema with specific data types
players_table_schema = """
CREATE TABLE players (
    player_id INTEGER PRIMARY KEY,
    player_name TEXT
);
"""

# player_stats_schema = """
# CREATE TABLE players_stats (
#     player_id CHAR(7),
#     player_name VARCHAR(100),*
#     current_team_name VARCHAR(100),*
#     posRank DECIMAL(4,2),
#     current_team_id VARCHAR(10),*
#     position VARCHAR(20),*
#     pro_team VARCHAR(50),*
#     points DECIMAL(6,2),
#     projected_points DECIMAL(6,2),
#     avg_points DECIMAL(6,2) NOT NULL,
#     projected_avg_points DECIMAL(6,2),
#     actual_pointsScored DECIMAL(7,2),
#     games_played INTEGER,
#     FOREIGN KEY (current_team_id) REFERENCES teams(team_id)
#     FOREIGN KEY(player_id) REFERENCES players(player_id),
#     PRIMARY KEY (player_id, year)
# );
# """

adp_schema = """
CREATE TABLE average_draft_position (
    adp_id INTEGER PRIMARY KEY AUTOINCREMENT,
    player_id CHAR(7),
    player_name VARCHAR(100),
    year INTEGER,
    team_name VARCHAR(50),
    POS VARCHAR(7),
    position STRING,
    pos_rank INTEGER,
    espn INTEGER,
    sleeper INTEGER,
    avg FLOAT,
    FOREIGN KEY (player_id) REFERENCES players(player_id)
);
"""
draft_table_schema = """
CREATE TABLE drafts (
    player_id CHAR(7) PRIMARY KEY,
    year INTEGER,
    position VARCHAR(10),
    overallPickNumber INTEGER,
    team_id VARCHAR(4),
    roundPickNumber INTEGER,
    id INTEGER,
    roundId INTEGER,
    autoDraftTypeId INTEGER,
    lineupSlotId INTEGER,
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
);
"""

team_table_schema = """
CREATE TABLE teams(
    team_id INTEGER PRIMARY KEY,
    year INTEGER,
    team_name VARCHAR(100),
    abbrev VARCHAR(10),
    division_id INTEGER,
    division_name VARCHAR(100),
    wins INTEGER,
    losses INTEGER,
    ties INTEGER,
    points_for FLOAT,
    points_against FLOAT,
    draft_projected_rank INTEGER,
    final_standing INTEGER
);
"""

# Execute the schema creation
with engine.begin() as conn:   # begin() auto-commits or rolls back
    conn.execute(text("DROP TABLE IF EXISTS drafts;"))
    conn.execute(text("DROP TABLE IF EXISTS average_draft_position;"))
    conn.execute(text("DROP TABLE IF EXISTS players_stats;"))
    conn.execute(text("DROP TABLE IF EXISTS teams;"))
    conn.execute(text("DROP TABLE IF EXISTS players;"))
    conn.execute(text("DROP TABLE IF EXISTS next_season_projections;"))


    conn.execute(text(players_table_schema))
    conn.execute(text(team_table_schema))
    conn.execute(text(players_stats_schema))
    conn.execute(text(adp_schema))
    conn.execute(text(draft_table_schema))
# Database tables created successfully
# - posts table with post_id as PRIMARY KEY
# - comments table with comment_id as PRIMARY KEY and post_id as FOREIGN KEY

### Populating the database

In [30]:
players_database_df.to_sql('players', engine, if_exists='replace', index=False)
stats_database_df.to_sql('players_stats', engine, if_exists='replace', index=False)
adp_database_df.to_sql('average_draft_position', engine, if_exists='append', index=False)
draft_database_df.to_sql('drafts', engine, if_exists='replace', index=False)
teams_database_df.to_sql('teams', engine, if_exists='replace', index=False)
next_season_database_df.to_sql('next_season_projections', engine, if_exists='replace', index=False)


print(f"✅ Database populated: ADP: {len(adp_database_df)}, Players: {len(players_df)}, Draft: {len(draft_database_df)}")

✅ Database populated: ADP: 1877, Players: 979, Draft: 1168
